# Phase 3: Advanced Validation & Logistic Regression
**Objective:** Implement K-Fold cross-validation, compare evaluation metrics, and build a Logistic Regression classifier from scratch.

### Step 1: The Implementation Pattern (Cross-Validation)
We strictly follow: Dataset -> split -> baseline -> model -> cross-validation -> metric summary -> residual/error analysis -> final test once[cite: 3].

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression, HuberRegressor, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Dataset (Simulating a Business Revenue Prediction Project)[cite: 3]
np.random.seed(42)
n_samples = 1000
marketing_spend = np.random.uniform(10000, 100000, n_samples)
sales_reps = np.random.randint(5, 50, n_samples)

# True Revenue Formula + Noise
revenue = (marketing_spend * 2.5) + (sales_reps * 15000) + np.random.normal(0, 50000, n_samples)

X = np.column_stack((marketing_spend, sales_reps))
y = revenue

# 2. Split Data (Lock away the final test set)[cite: 3]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. K-Fold Cross-Validation (Running 5 separate training views)[cite: 3]
model = LinearRegression()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# We use cross_validate to get multiple metrics at once
cv_results = cross_validate(model, X_train, y_train, cv=kf, 
                            scoring=('neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2'))

# 4. Compare mean and spread of scores[cite: 3]
# We take the absolute value because sklearn returns negative errors for optimization compatibility
mae_scores = np.abs(cv_results['test_neg_mean_absolute_error'])
rmse_scores = np.abs(cv_results['test_neg_root_mean_squared_error'])
r2_scores = cv_results['test_r2']

print("--- K-FOLD CROSS-VALIDATION RESULTS (5 Folds) ---")
print(f"MAE:  Mean = ${mae_scores.mean():,.2f} | Spread (Std) = ${mae_scores.std():,.2f}")
print(f"RMSE: Mean = ${rmse_scores.mean():,.2f} | Spread (Std) = ${rmse_scores.std():,.2f}")
print(f"R^2:  Mean = {r2_scores.mean():.4f}  | Spread (Std) = {r2_scores.std():.4f}")

--- K-FOLD CROSS-VALIDATION RESULTS (5 Folds) ---
MAE:  Mean = $39,898.31 | Spread (Std) = $23.25
RMSE: Mean = $50,006.34 | Spread (Std) = $26.12
R^2:  Mean = 0.9440  | Spread (Std) = 0.0000


### Step 2: The "Good R² but Unacceptable Errors" Trap
**Objective:** Create a model that has good $R^2$ but unacceptable large errors and investigate why[cite: 3].

In [36]:
# Train the final model on the full training set
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

final_r2 = r2_score(y_test, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mean_revenue = np.mean(y_test)

print("--- FINAL EVALUATION ON TEST SET ---")
print(f"Average Business Revenue: ${mean_revenue:,.2f}")
print(f"Final R^2 Score: {final_r2:.4f}")
print(f"Final RMSE: ${final_rmse:,.2f}")
print("-" * 50)
print("INVESTIGATION:")
print("Notice that our R^2 is incredibly high (nearly 0.90+). A junior developer would call this 'perfect'.")
print(f"However, the RMSE is around ${final_rmse:,.0f}.")
print("If the business operates on a razor-thin 5% profit margin, an absolute error of $50,000 in revenue forecasting could mean the difference between hiring new staff or going bankrupt.")
print("Conclusion: R^2 tells you the mathematical fit. RMSE tells you the actual business impact. You must compare MAE, RMSE and R^2 on the same predictions[cite: 3].")

--- FINAL EVALUATION ON TEST SET ---
Average Business Revenue: $509,976.58
Final R^2 Score: 0.9503
Final RMSE: $47,352.36
--------------------------------------------------
INVESTIGATION:
Notice that our R^2 is incredibly high (nearly 0.90+). A junior developer would call this 'perfect'.
However, the RMSE is around $47,352.
If the business operates on a razor-thin 5% profit margin, an absolute error of $50,000 in revenue forecasting could mean the difference between hiring new staff or going bankrupt.
Conclusion: R^2 tells you the mathematical fit. RMSE tells you the actual business impact. You must compare MAE, RMSE and R^2 on the same predictions[cite: 3].


### Step 3: Logistic Regression Math Engine (From Scratch)
We will now build the classification engine using the exact formulas from Phase 1. 
1. The Sigmoid Function to squash linear scores into probabilities.
2. The Binary Cross-Entropy (Log Loss) function.
3. The Gradient Descent update loop.

In [37]:
# 1. The Mathematical Functions
def sigmoid(z):
    # Clips z to prevent math overflow errors on massive numbers
    z = np.clip(z, -250, 250)
    return 1 / (1 + np.exp(-z))

def compute_log_loss(y, p):
    # Adds a tiny epsilon (1e-15) so we never calculate log(0), which crashes the system
    epsilon = 1e-15
    p = np.clip(p, epsilon, 1 - epsilon)
    # L = -[y*log(p) + (1-y)*log(1-p)]
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# 2. The Gradient Descent Training Loop
def train_logistic_regression(X, y, alpha=0.01, iterations=1000):
    m = len(y)
    w = np.zeros(X.shape[1])
    b = 0.0
    loss_history = []
    
    for i in range(iterations):
        # Step A: Linear Score (z = Xw + b)
        z = np.dot(X, w) + b
        
        # Step B: Pass through Sigmoid to get Probability (p)
        p = sigmoid(z)
        
        # Step C: Calculate Gradients (Identical to Linear Regression!)
        dw = (1/m) * np.dot(X.T, (p - y))
        db = (1/m) * np.sum(p - y)
        
        # Step D: Update Weights
        w = w - (alpha * dw)
        b = b - (alpha * db)
        
        # Record Log Loss
        loss_history.append(compute_log_loss(y, p))
        
    return w, b, loss_history

print("Logistic Regression functions compiled successfully. The engine is ready.")

Logistic Regression functions compiled successfully. The engine is ready.
